In [ ]:
'''# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session'''

In [2]:
import numpy as np
import pandas as pd


In [3]:
df=pd.read_csv('/kaggle/input/datasets/adithip2000/genre-generalised-and-tagged/book_movie_combined.csv')

In [4]:
df.head()

,Unnamed: 0,summary,genres,form,mapped_genres
0,0,"Old Major, the old boar on the Manor Farm, ca...","['sci-fi', 'drama', 'family']",book,NaN
1,1,"Alex, a teenager living in near-future Englan...","['sci-fi', 'drama']",book,NaN
2,2,The text of The Plague is divided into five p...,['drama'],book,NaN
3,3,The novel posits that space around the Milky ...,"['sci-fi', 'fantasy', 'drama']",book,NaN
4,4,"Ged is a young boy on Gont, one of the larger...","['sci-fi', 'fantasy', 'drama', 'family']",book,NaN


In [5]:
df.shape

(50048, 5)

## Removing unnecessary columns 

In [8]:
df.drop(columns=['mapped_genres'],inplace=True)

In [12]:
df.drop(columns=['Unnamed: 0'],inplace=True)

In [13]:
df.head()

,summary,genres,form
0,"Old Major, the old boar on the Manor Farm, ca...","['sci-fi', 'drama', 'family']",book
1,"Alex, a teenager living in near-future Englan...","['sci-fi', 'drama']",book
2,The text of The Plague is divided into five p...,['drama'],book
3,The novel posits that space around the Milky ...,"['sci-fi', 'fantasy', 'drama']",book
4,"Ged is a young boy on Gont, one of the larger...","['sci-fi', 'fantasy', 'drama', 'family']",book


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50048 entries, 0 to 50047
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   summary  50048 non-null  object
 1   genres   50048 non-null  object
 2   form     50048 non-null  object
dtypes: object(3)
memory usage: 1.1+ MB


In [23]:
df1=df.copy()

## Fix escaped characters (Alex\'s----> Alex's)


## Fix html entities 

## &amp; → &

## &quot; → "

## &lt; → <

## normalize spaces(Removes extra space, newlines,tabs)

## Converted to lower case (use bert-base-uncased)



In [25]:
import re
import html
def clean_text(text):
    #Fix escaped character
    text = text.replace("\\'", "'")
    text = text.replace('\\"', '"')
    text = text.replace("\\n", " ")
    text = text.replace("\\t", " ")
    #Fix HTML entities
    text=html.unescape(text)
    #Normalize spaces
    text = re.sub(r'\s+', ' ', text)
    #Lowercase
    text = text.lower()
    return text.strip()
    

In [26]:
df['summary_clean']=df['summary'].apply(clean_text)

In [33]:
df.drop(columns=['summary'],inplace=True)

In [34]:
df.head(10)

,genres,form,summary_clean
0,"['sci-fi', 'drama', 'family']",book,"old major, the old boar on the manor farm, cal..."
1,"['sci-fi', 'drama']",book,"alex, a teenager living in near-future england..."
2,['drama'],book,the text of the plague is divided into five pa...
3,"['sci-fi', 'fantasy', 'drama']",book,the novel posits that space around the milky w...
4,"['sci-fi', 'fantasy', 'drama', 'family']",book,"ged is a young boy on gont, one of the larger ..."
5,['sci-fi'],book,"living on mars, deckard is acting as a consult..."
6,['sci-fi'],book,beginning several months after the events in b...
7,"['sci-fi', 'drama']",book,the story is told through the eyes of narrator...
8,"['sci-fi', 'drama', 'family']",book,nine years after emperor paul muad'dib walked ...
9,"['sci-fi', 'drama', 'family']",book,the situation is desperate for the bene gesser...


In [31]:
print(df.iloc[1]['summary_clean'])

alex, a teenager living in near-future england, leads his gang on nightly orgies of opportunistic, random "ultra-violence." alex's friends ("droogs" in the novel's anglo-russian slang, nadsat) are: dim, a slow-witted bruiser who is the gang's muscle; georgie, an ambitious second-in-command; and pete, who mostly plays along as the droogs indulge their taste for ultra-violence. characterized as a sociopath and a hardened juvenile delinquent, alex is also intelligent and quick-witted, with sophisticated taste in music, being particularly fond of beethoven, or "lovely ludwig van." the novel begins with the droogs sitting in their favorite hangout (the korova milkbar), drinking milk-drug cocktails, called "milk-plus", to hype themselves for the night's mayhem. they assault a scholar walking home from the public library, rob a store leaving the owner and his wife bloodied and unconscious, stomp a panhandling derelict, then scuffle with a rival gang. joyriding through the countryside in a sto

## Removing {{plot}} tag

In [37]:
df[df['summary_clean'].str.contains(r'\{\{plot\}\}', case=False, na=False)]

,genres,form,summary_clean
12110,"['thriller', 'drama', 'horror']",movie,"{{plot}} the film opens in 1974, as a young gi..."
12118,['comedy'],movie,{{plot}} following the sudden death of kid's f...
12158,['horror'],movie,{{plot}} the film begins with a boy named mich...
12159,"['drama', 'romance']",movie,{{plot}} husband and wife rawdon and miranda a...
12270,['action'],movie,{{plot}} in the dasht-e-margoh desert in remot...
...,...,...,...
49620,"['comedy', 'drama']",movie,{{plot}} the film focuses on various guests st...
49633,"['horror', 'mystery']",movie,{{plot}} gp: guard post is on the frontline in...
49702,"['sci-fi', 'fantasy', 'drama']",movie,{{plot}} news reporter ichiro sakai and photog...
49779,"['sci-fi', 'comedy', 'family']",movie,{{plot}} owen baker is a 12-year-old who has b...


In [42]:
import re

def remove_plot_tag(text):
    # remove {{plot}} (case insensitive)
    text = re.sub(r'\{\{\s*plot\s*\}\}', '', text, flags=re.IGNORECASE)
    return text

In [43]:
df['summary_clean'] = df['summary_clean'].apply(remove_plot_tag)

## Detect other templates like {{....}} 

## {{plot}} ,{{summary}},...

In [47]:
df['summary_clean'].str.contains(r'\{\{.*?\}\}', na=False).sum()

np.int64(2171)

In [49]:
df[df['summary_clean'].str.contains(r'\{\{.*?\}\}', na=False)].head()

,genres,form,summary_clean
12169,"['thriller', 'horror']",movie,a mexican widow receives a letter from the hig...
12172,['drama'],movie,{{expand section}} the film follows the bandin...
12183,"['comedy', 'drama', 'romance']",movie,it is spring. randy dean is a high school seni...
12210,['thriller'],movie,the story centers around johnny graham who was...
12239,['drama'],movie,two families looking to escape painful holiday...


In [50]:
#extract templates


def extract_templates(text):
    return re.findall(r'\{\{.*?\}\}', text)

df['templates'] = df['summary_clean'].apply(extract_templates)

In [55]:
all_templates = df['templates'].explode().dropna().unique()
print(all_templates)

['{{quote box}}' '{{expand section}}' '{{cquote}}' '{{cite web}}'
 '{{long plot}}' '{{cite news}}' '{{cite book}}'
 '{{plot|date"farewell my concubine study notes">{{cite web}}'
 '{{citation needed}}'
 '{{cite book |last1frank |title1974 |publisher0-913460-31-1 |page"cohen-78">{{cite book}}'
 '{{ref_label}}' '{{fact}}' '{{quote}}' '{{mdash}}'
 '{{inappropriate tone}}' '{{see also}}' '{{inr}}' '{{cite journal}}'
 '{{spaced ndash}}' '{{uss}}' '{{nihongo}}' '{{citation}}'
 '{{plot|date"mash4077couk">classic episode - goodbye, farewell and amen. mash4077.co.uk. in pierce’s first recollection, he was on a bus returning to the 4077th after a day of drinking at the beaches of incheon. he called for a bottle of whiskey to be passed back to someone who “can’t wait”; later, he is able to more accurately recall this person was a wounded soldier, and that the bottle was filled with not whiskey, but plasma. the bus then picked up some south korean refugees, followed by some wounded soldiers who bro

In [57]:
#removing only simple templates {{cite}},{{cleanup}}
def remove_simple_templates(text):
    return re.sub(r'\{\{\s*[a-zA-Z ]+\s*\}\}', '', text)

In [58]:
df['summary_clean']=df['summary_clean'].apply(remove_simple_templates)

In [60]:
df['summary_clean'].str.contains(r'\{\{.*?\}\}', na=False).sum()

np.int64(115)

In [61]:
df['templates'] = df['summary_clean'].apply(extract_templates)
all_templates = df['templates'].explode().dropna().unique()
print(all_templates)

['{{ref_label}}'
 '{{plot|date"mash4077couk">classic episode - goodbye, farewell and amen. mash4077.co.uk. in pierce’s first recollection, he was on a bus returning to the 4077th after a day of drinking at the beaches of incheon. he called for a bottle of whiskey to be passed back to someone who “can’t wait”; later, he is able to more accurately recall this person was a wounded soldier, and that the bottle was filled with not whiskey, but plasma. the bus then picked up some south korean refugees, followed by some wounded soldiers who brought news of an enemy patrol in the area. the bus later pulls off the road and everyone is told to stay quiet so they would not be discovered by the enemy. one woman carried a live chicken that would not stop squawking, prompting pierce to angrily admonish her to “keep that damn chicken quiet!”, after which the noise suddenly stopped. this last detail causes pierce to break down sobbing as he finally reveals the true ending of the story. when hawkeye sn

In [63]:
#extract plot text {{plot |....}}
def extract_plot_content(text):
    match = re.search(r'\{\{plot\|.*?"(.*?)\}\}', text, flags=re.DOTALL)
    if match:
        return match.group(1)
    return text

In [64]:
df['summary_clean'] = df['summary_clean'].apply(extract_plot_content)

In [66]:
#remove remaining templates
def remove_all_templates(text):
    return re.sub(r'\{\{.*?\}\}', '', text)

In [67]:
df['summary_clean'] = df['summary_clean'].apply(remove_all_templates)

In [68]:
df['summary_clean'].str.contains(r'\{\{.*?\}\}', na=False).sum()

np.int64(0)

In [69]:
df['templates'] = df['summary_clean'].apply(extract_templates)
all_templates = df['templates'].explode().dropna().unique()
print(all_templates)

[]


In [118]:
#check if any summary is empty
(df['summary_clean'].str.strip() == '').sum()

np.int64(0)

In [75]:
print(df['summary_clean'].iloc[12239])

two families looking to escape painful holiday memories take a vacation to an exotic caribbean island over christmas. a widowed mother, dana , and her two sons, chris and michael meet a divorced father, dan ([[colin ferguson , and his two daughters, blair and nell when their cruise ship docks in puerto rico. despite a rough start, the parents and kids begin to develop bonds over the course of their stay at a beautiful puerto rican beach resort. it looks like an idyllic brady bunch holiday for the two families as dana and dan try to put their personal tragedies behind them and begin to grow closer, until an unexpected visitor from the past appears and threatens their tentative romance. what promised to be a joyous christmas filled with fresh hope and new relationships turns complicated as each member of the two families must sort out their feelings and choose their own path.


In [78]:
df.drop(columns=['templates'],inplace=True)

In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50048 entries, 0 to 50047
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   genres         50048 non-null  object
 1   form           50048 non-null  object
 2   summary_clean  50048 non-null  object
dtypes: object(3)
memory usage: 1.1+ MB


,genres,form,summary_clean
33217,"['thriller', 'drama', 'crime']",movie,former cop billy taggart begins following the ...
22636,['sci-fi'],movie,"in a remote arctic research station, governmen..."
9726,"['sci-fi', 'drama']",book,"the tale begins with two college-age brothers,..."
36952,"['action', 'adventure']",movie,"an aging, respected commander paul blanchard, ..."
2047,['romance'],book,"the son of the duke and duchess of avon, the m..."
9133,"['drama', 'romance']",book,harry silver is a successful television produc...
9137,['sci-fi'],book,"the main character, sir charles ravenstreet, i..."
23839,"['action', 'drama', 'family']",movie,"sora, umi and ao are three school-aged girls w..."
17083,"['sci-fi', 'action']",movie,{{ref improve section|datepart onephoto of mai...
8152,"['sci-fi', 'thriller', 'drama', 'mystery']",book,~plot outline description~ --> <!--


In [85]:
print(df.iloc[17083]['summary_clean'])

{{ref improve section|datepart onephoto of main characters|the first two primary characters of half-life: escape from city 17, potrayed by derek chan and ian purchase. the citadel can be seen in the background.]] --> part one is set during the second half of events of half-life 2: episode one. dr. isaac kleiner is making his "kleinercasts" on city 17's pa system, warning that the combine citadel is set to explode at any moment; should the citadel detonate, the resulting explosion will destroy the city and the surrounding area. prior to the film's events, gordon freeman's actions within the citadel have held down the impending explosion, opening a small window of time for civilians to escape. members of the lambda resistance are seen fighting their way out of the city as the combine's civil protection forces try to hold them back. cp officers are also seen executing captured rebels, while combine synths wreak havoc on the warzone. two male resistance members are introduced escaping thro

In [86]:
print(df.iloc[8152]['summary_clean'])

~plot outline description~ --> <!--


## Checking problematic rows  {{ ......

In [88]:
mask = df['summary_clean'].str.startswith('{{', na=False)
print(mask.sum())

66


In [90]:
df[mask]['form'].value_counts()

form
movie    66
Name: count, dtype: int64

In [91]:
df[mask]['genres'].explode().value_counts().head()

genres
['drama']               5
['comedy']              4
['drama', 'romance']    3
['comedy', 'drama']     3
['comedy', 'family']    3
Name: count, dtype: int64

In [93]:
# only 66 rows are there so removing them 

mask = df['summary_clean'].str.startswith('{{', na=False)
df = df[~mask]

In [94]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49982 entries, 0 to 50047
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   genres         49982 non-null  object
 1   form           49982 non-null  object
 2   summary_clean  49982 non-null  object
dtypes: object(3)
memory usage: 1.5+ MB


In [95]:
df.shape

(49982, 3)

## Checking HTML comments /summary starting with plot outline and removing them

In [115]:
#25 rows removed
mask = (
    df['summary_clean'].str.contains('plot outline', case=False, na=False) |
    df['summary_clean'].str.contains(r'<!--.*?-->', na=False)
)

df = df[~mask]

In [117]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49957 entries, 0 to 50047
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   genres         49957 non-null  object
 1   form           49957 non-null  object
 2   summary_clean  49957 non-null  object
dtypes: object(3)
memory usage: 1.5+ MB


In [122]:
df.sample(10)

,genres,form,summary_clean
9966,['sci-fi'],book,the story follows the heroic jedi knights as t...
12517,"['sci-fi', 'thriller', 'action']",movie,houston cop jack caine does not let police pro...
34245,['comedy'],movie,italian election day in the early '80s. three ...
42525,"['sci-fi', 'action', 'family']",movie,green legend is set in a science fiction-style...
3906,"['thriller', 'drama', 'mystery', 'crime']",book,stephanie is assigned to bring in semi-retired...
24479,"['thriller', 'drama', 'mystery']",movie,jennifer peters attempts to save her brother r...
23917,"['action', 'drama', 'romance', 'comedy']",movie,"in 1914, american showgirl ""suzy"" trent is in ..."
35435,['comedy'],movie,duane balfour ([[douglas smith was born into a...
43012,['crime'],movie,ed hutcheson is the crusading managing editor ...
1136,"['drama', 'family']",book,"gene forrester, the protagonist, returns to hi..."


In [120]:
df.to_csv("clean_dataset.csv", index=False, encoding='utf-8')